In [ ]:
# Install runtime dependencies (unpinned, Colab-friendly)
!pip install -q trl peft tensorboard bitsandbytes datasets transformers accelerate

In [ ]:
import gc
import os
import re
import random
from decimal import Decimal, InvalidOperation

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

# ── Fixed choices (all user-adjustable settings in one place) ──────────────
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "openai/gsm8k"
DATASET_CONFIG = "main"
SOURCE_SPLIT = "train"
SEED = 42
HOLDOUT_FRACTION = 0.1
RL_SUBSET_SIZE = 64
EVAL_SUBSET_SIZE = 64
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
MAX_STEPS = 16
LEARNING_RATE = 1e-5
NUM_GENERATIONS = 8
TEMPERATURE = 0.9
TOP_P = 0.95
MAX_PROMPT_LENGTH = 512
MAX_COMPLETION_LENGTH = 256
TB_LOG_DIR = "/content/tb_grpo_sanity"

# ── Seed everything ────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── CUDA gate ──────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), (
    "CUDA is not available. This notebook requires a GPU runtime (e.g. Colab T4)."
)

print(f"Device       : {torch.cuda.get_device_name(0)}")
print(f"CUDA version : {torch.version.cuda}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {__import__('transformers').__version__}")
print(f"TRL          : {__import__('trl').__version__}")
print(f"PEFT         : {__import__('peft').__version__}")

In [ ]:
# Load GSM8K training data and create the deterministic 10% holdout
# before selecting the 64-example RL subset.
raw_ds = load_dataset(DATASET_ID, DATASET_CONFIG, split=SOURCE_SPLIT)
print(f"Full source split size: {len(raw_ds)}")

# Deterministic 90/10 split
split = raw_ds.train_test_split(test_size=HOLDOUT_FRACTION, seed=SEED)
train_pool = split["train"]   # 90% for RL training pool
holdout = split["test"]       # 10% held out for evaluation

# Deterministic 64-example RL subset from training pool
assert RL_SUBSET_SIZE <= len(train_pool), (
    f"RL_SUBSET_SIZE={RL_SUBSET_SIZE} exceeds training pool size {len(train_pool)}"
)
rl_indices = np.random.RandomState(SEED).choice(len(train_pool), size=RL_SUBSET_SIZE, replace=False)
rl_data = train_pool.select(sorted(rl_indices))

print(f"Training pool : {len(train_pool)}")
print(f"Holdout       : {len(holdout)}")
print(f"RL subset     : {len(rl_data)}")

In [ ]:
# Select the 64-example evaluation subset from the holdout
assert EVAL_SUBSET_SIZE <= len(holdout), (
    f"EVAL_SUBSET_SIZE={EVAL_SUBSET_SIZE} exceeds holdout size {len(holdout)}"
)
eval_indices = np.random.RandomState(SEED).choice(len(holdout), size=EVAL_SUBSET_SIZE, replace=False)
eval_data = holdout.select(sorted(eval_indices))
print(f"Eval subset   : {len(eval_data)}")

In [ ]:
# Load tokenizer and 4-bit quantized model with device_map="auto"
if "model" in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Set EOS pad token when needed and use left padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

# 4-bit NF4 quantization, double quantization, FP16 compute
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded: {MODEL_ID} (4-bit NF4)")
print(f"Pad token id: {tokenizer.pad_token_id}")
print(f"Device map  : {getattr(model, 'hf_device_map', 'N/A')}")

In [ ]:
# Baseline Qwen chat-template prompt and numeric answer parsing helpers
SYSTEM_PROMPT = (
    "Solve the following math problem step by step. "
    "Show your reasoning, then finish with the final numeric answer on a new line in this exact format:\n"
    "#### <numeric answer>\n"
    "Do not include any text after the #### line."
)


def build_messages(question: str) -> list[dict]:
    """Return chat messages for a GSM8K question."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]


def apply_prompt(messages: list[dict]) -> str:
    """Apply the Qwen chat template with a generation prompt."""
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


def normalize_numeric(text: str) -> str:
    """Strip whitespace, commas, currency symbols, trailing punctuation."""
    text = text.strip()
    text = re.sub(r"[\$,]", "", text)
    text = text.rstrip(". !?")
    return text.strip()


def extract_gold_numeric(answer_text: str) -> str:
    """Extract the numeric answer after the final #### in GSM8K answer."""
    match = re.search(r"####\s*(.+)$", answer_text.strip(), re.MULTILINE)
    return match.group(1).strip() if match else ""


def extract_predicted_answer(generated_text: str) -> tuple[str, str]:
    """Extract predicted answer; prefer #### value, fall back to last numeric token.
    Returns (predicted_answer, parse_status).
    """
    match = re.search(r"####\s*(.+)$", generated_text.strip(), re.MULTILINE)
    if match:
        raw = match.group(1).strip()
        normalized = normalize_numeric(raw)
        if normalized:
            return normalized, "####_match"

    numbers = re.findall(r"-?\d[\d,]*\.?\d*", generated_text)
    if numbers:
        normalized = normalize_numeric(numbers[-1])
        if normalized:
            return normalized, "fallback_numeric"

    return "", "parse_failure"


def answers_match(predicted: str, gold: str) -> bool:
    """Compare predicted and gold answers using decimal-safe parsing."""
    pred_norm = normalize_numeric(predicted)
    gold_norm = normalize_numeric(gold)
    if not pred_norm or not gold_norm:
        return False
    try:
        return Decimal(pred_norm) == Decimal(gold_norm)
    except (InvalidOperation, ValueError):
        return pred_norm == gold_norm


# Quick sanity check
test_messages = build_messages("What is 2 + 3?")
test_prompt = apply_prompt(test_messages)
print(f"Prompt template OK — length: {len(test_prompt)} chars")
print(f"Sample gold extraction: '#### 42' -> '{extract_gold_numeric('#### 42')}'")

In [ ]:
# Convert examples to the prompt/answer structure expected by GRPOTrainer.
# The dataset needs a "prompt" column (list of message dicts) and an "answer" column
# that will be forwarded to reward functions as a keyword argument.

def format_example(example: dict) -> dict:
    return {
        "prompt": build_messages(example["question"]),
        "answer": extract_gold_numeric(example["answer"]),
    }

rl_dataset = rl_data.map(format_example, remove_columns=rl_data.column_names)
eval_dataset = eval_data.map(format_example, remove_columns=eval_data.column_names)

print(f"RL dataset columns: {rl_dataset.column_names}")
print(f"RL dataset size   : {len(rl_dataset)}")
print(f"Eval dataset size : {len(eval_dataset)}")
print(f"Sample prompt type: {type(rl_dataset[0]['prompt'])}")
print(f"Sample answer     : {rl_dataset[0]['answer']}")

In [ ]:
# Define correctness and format reward functions.
# Both return one value per completion and accept extra keyword arguments for TRL compatibility.
# The "answer" column from the dataset is passed via kwargs.
#
# TRL returns completions as message lists (list[dict[str, str]]), not raw strings.
# The _to_text helper normalizes both formats so reward logic always works on strings.

def _to_text(completion) -> str:
    \"\"\"Extract text from a TRL completion (message list or raw string).\"\"\"
    if isinstance(completion, list) and len(completion) > 0 and isinstance(completion[0], dict):
        return completion[0].get("content", "")
    return str(completion)


def correctness_reward(completions, answer: list[str], **kwargs) -> list[float]:
    \"\"\"1.0 when parsed generated answer matches gold, else 0.0.\"\"\"
    rewards = []
    for completion, gold in zip(completions, answer):
        text = _to_text(completion)
        predicted, _ = extract_predicted_answer(text)
        rewards.append(1.0 if answers_match(predicted, gold) else 0.0)
    return rewards


def format_reward(completions, **kwargs) -> list[float]:
    \"\"\"0.1 when completion contains a valid #### <number> answer at end of line, else 0.0.\"\"\"
    rewards = []
    for completion in completions:
        text = _to_text(completion)
        # Anchor to end-of-line: no trailing text allowed after the number
        match = re.search(r"####\s*(-?[\d,]+\.?\d*)\s*$", text, re.MULTILINE)
        rewards.append(0.1 if match else 0.0)
    return rewards


# Quick test (raw strings)
test_completions = ["The answer is\n#### 42", "I don't know", "#### 3.14 extra text"]
test_answers = ["42", "7", "3.14"]
print("Correctness (strings):", correctness_reward(test_completions, answer=test_answers))
print("Format      (strings):", format_reward(test_completions))

# Quick test (message lists, as TRL returns during training)
test_msg_completions = [
    [{"role": "assistant", "content": "The answer is\n#### 42"}],
    [{"role": "assistant", "content": "I don't know"}],
    [{"role": "assistant", "content": "#### 3.14 extra text"}],
]
print("Correctness (messages):", correctness_reward(test_msg_completions, answer=test_answers))
print("Format      (messages):", format_reward(test_msg_completions))

In [ ]:
# Configure GRPOConfig and GRPOTrainer with LoRA PEFT configuration,
# the RL data, and held-out eval data.

training_args = GRPOConfig(
    output_dir="/content/grpo_output",
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    num_generations=NUM_GENERATIONS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    logging_steps=1,
    report_to="tensorboard",
    logging_dir=TB_LOG_DIR,
    save_strategy="no",
    save_total_limit=0,
    seed=SEED,
    remove_unused_columns=False,
    gradient_checkpointing=False,
)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=[correctness_reward, format_reward],
    train_dataset=rl_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("GRPOTrainer configured.")
print(f"  Max steps        : {MAX_STEPS}")
print(f"  Num generations  : {NUM_GENERATIONS}")
print(f"  Batch size       : {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"  Grad accum       : {GRADIENT_ACCUMULATION_STEPS}")
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  TensorBoard dir  : {TB_LOG_DIR}")

In [ ]:
# Train for 16 steps with saving disabled and TensorBoard logging enabled.
trainer.train()

# Validate training completed the expected number of steps
assert trainer.state.global_step == MAX_STEPS, (
    f"Expected {MAX_STEPS} steps, got {trainer.state.global_step}"
)

# Verify logged reward metrics are finite
log_history = trainer.state.log_history
reward_logs = [entry for entry in log_history if "reward" in entry]
for entry in reward_logs:
    for key, val in entry.items():
        if isinstance(val, (int, float)):
            assert np.isfinite(val), f"Non-finite value in log: {key}={val}"

print(f"\nTraining complete: {trainer.state.global_step} steps verified.")
print(f"Reward log entries: {len(reward_logs)}")

In [ ]:
# Post-training evaluation on 64 held-out examples.
# Generate NUM_GENERATIONS completions per example (matching training config).
eval_model = trainer.model
eval_model.eval()

eval_results = []       # one entry per completion (64 * 8 = 512 total)
eval_per_example = []   # one entry per example (64 total)

for i in range(len(eval_dataset)):
    messages = eval_dataset[i]["prompt"]
    gold = eval_dataset[i]["answer"]
    prompt_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH).to(eval_model.device)

    with torch.no_grad():
        output_ids = eval_model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION_LENGTH,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            num_return_sequences=NUM_GENERATIONS,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_ids = output_ids[:, input_len:]
    completions = tokenizer.batch_decode(new_ids, skip_special_tokens=True)

    example_correct = False
    example_rewards = []

    for comp_text in completions:
        predicted, parse_status = extract_predicted_answer(comp_text)
        correct = answers_match(predicted, gold)
        corr_reward = 1.0 if correct else 0.0
        fmt_reward = 0.1 if re.search(r"####\s*(-?[\d,]+\.?\d*)\s*$", comp_text, re.MULTILINE) else 0.0
        total = corr_reward + fmt_reward

        eval_results.append({
            "example_idx": i,
            "question": messages[1]["content"],
            "gold": gold,
            "completion": comp_text,
            "predicted": predicted,
            "parse_status": parse_status,
            "correct": correct,
            "correctness_reward": corr_reward,
            "format_reward": fmt_reward,
            "total_reward": total,
        })

        example_rewards.append(total)
        if correct:
            example_correct = True

    eval_per_example.append({
        "idx": i,
        "gold": gold,
        "any_correct": example_correct,
        "mean_reward": sum(example_rewards) / len(example_rewards),
    })

    del inputs, output_ids, new_ids

print(f"Evaluated {len(eval_dataset)} examples x {NUM_GENERATIONS} completions = {len(eval_results)} total.")

In [ ]:
# Print aggregate reward/accuracy/parse metrics
# and three prompt/completion examples with reward components.

num_completions = len(eval_results)
num_examples = len(eval_per_example)
correct_count = sum(1 for r in eval_results if r["correct"])
parse_failures = sum(1 for r in eval_results if r["parse_status"] == "parse_failure")
accuracy = correct_count / num_completions if num_completions > 0 else 0.0
avg_reward = sum(r["total_reward"] for r in eval_results) / num_completions if num_completions > 0 else 0.0
avg_correctness = sum(r["correctness_reward"] for r in eval_results) / num_completions if num_completions > 0 else 0.0
avg_format = sum(r["format_reward"] for r in eval_results) / num_completions if num_completions > 0 else 0.0
parse_rate = (num_completions - parse_failures) / num_completions if num_completions > 0 else 0.0
pass_at_1_group = sum(1 for ex in eval_per_example if ex["any_correct"]) / num_examples if num_examples > 0 else 0.0

# Verify reward value domains
all_correctness = [r["correctness_reward"] for r in eval_results]
all_format = [r["format_reward"] for r in eval_results]
assert set(all_correctness) <= {0.0, 1.0}, f"Correctness values outside {{0.0, 1.0}}: {set(all_correctness)}"
assert set(all_format) <= {0.0, 0.1}, f"Format values outside {{0.0, 0.1}}: {set(all_format)}"
assert all(np.isfinite(r["total_reward"]) for r in eval_results), "Non-finite reward detected"

print("=" * 60)
print("AGGREGATE METRICS")
print("=" * 60)
print(f"Model            : {MODEL_ID}")
print(f"Dataset          : {DATASET_ID} ({DATASET_CONFIG})")
print(f"Training steps   : {MAX_STEPS}")
print(f"Examples         : {num_examples}")
print(f"Completions      : {num_completions} ({num_examples} x {NUM_GENERATIONS})")
print(f"Correct          : {correct_count}")
print(f"Accuracy         : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Pass@{NUM_GENERATIONS} (any correct): {pass_at_1_group:.4f} ({pass_at_1_group * 100:.2f}%)")
print(f"Parse rate       : {parse_rate:.4f}")
print(f"Parse failures   : {parse_failures}")
print(f"Avg total reward : {avg_reward:.4f}")
print(f"Avg correctness  : {avg_correctness:.4f}")
print(f"Avg format       : {avg_format:.4f}")
print("=" * 60)

# Three diagnostic examples with reward components (one completion per example)
print("\nSAMPLE COMPLETIONS (3 examples, first completion each):")
print("-" * 60)
shown = 0
for ex in eval_per_example[:3]:
    # Find first completion for this example
    r = next(r for r in eval_results if r["example_idx"] == ex["idx"])
    status_mark = "✓" if r["correct"] else "✗"
    any_mark = "✓" if ex["any_correct"] else "✗"
    print(f"[{status_mark}] Example {ex['idx']}  (any_correct={any_mark}, mean_reward={ex['mean_reward']:.2f})")
    print(f"    Q       : {r['question'][:120]}{'...' if len(r['question']) > 120 else ''}")
    print(f"    Gold    : {r['gold']}")
    print(f"    Predicted: {r['predicted']}  (parse: {r['parse_status']})")
    print(f"    Correctness reward: {r['correctness_reward']}")
    print(f"    Format reward     : {r['format_reward']}")
    print(f"    Total reward      : {r['total_reward']}")
    comp_preview = r['completion'].replace('\n', ' ')[:200]
    print(f"    Completion: {comp_preview}{'...' if len(r['completion'].replace(chr(10), ' ')) > 200 else ''}")
    print()

In [ ]:
# Verify TensorBoard event files exist and print the log directory/viewing command.
import glob as glob_mod

tb_files = glob_mod.glob(os.path.join(TB_LOG_DIR, "**", "events.out.tfevents.*"), recursive=True)
print(f"TensorBoard log directory: {TB_LOG_DIR}")
print(f"Event files found: {len(tb_files)}")
for f in tb_files[:5]:
    print(f"  {f}")

assert len(tb_files) > 0, "No TensorBoard event files found!"
print(f"\nTo view in Colab:")
print(f"  %load_ext tensorboard")
print(f"  %tensorboard --logdir {TB_LOG_DIR}")
print(f"\nOr from terminal:")
print(f"  tensorboard --logdir {TB_LOG_DIR}")